In [ ]:
from netCDF4 import Dataset
import numpy as np
import os
import pandas as pd
from matplotlib import pyplot as plt
import fnmatch

In [ ]:
from mpl_toolkits.basemap import Basemap

In [ ]:
def find_files(directory, pattern, maxdepth=None):
    flist = []
    for root, dirs, files in os.walk(directory):
        for basename in files:
            if fnmatch.fnmatch(basename, pattern):
                filename = os.path.join(root, basename)
                filename = filename.replace('\\\\', os.sep)
                if maxdepth is None:
                    flist.append(filename)
                else:
                    if filename.count(os.sep)-directory.count(os.sep) <= maxdepth:
                        flist.append(filename)
    return flist

In [ ]:
files = find_files('/mnt/hippocamp/DATA/sattelite/AQUARIUS/V5/2012/', '*')

In [ ]:
files[700]

In [ ]:
data = Dataset(f'{files[0]}', 'r')

In [ ]:
data.variables

In [ ]:
import xarray as xr
ds = xr.open_dataset(f'{files[700]}')
print(ds)

In [ ]:
ds.variables

In [ ]:
nc = Dataset(f'{files[0]}', 'r')

In [ ]:
print("data_model:", nc.data_model)

In [ ]:
print("Groups:", list(nc.groups.keys()))

In [ ]:
print("Root variables:", list(nc.variables.keys()))

In [ ]:
for gname, grp in nc.groups.items():
    print("Group:", gname)

In [ ]:
ds = xr.open_dataset(f'{files[0]}', group='Navigation', engine='netcdf4')
print(ds)
print(ds.data_vars)
print(ds.coords)

In [ ]:
ds = xr.open_dataset(f'{files[0]}', group='Aquarius Data', engine='netcdf4')
print(ds)
print(ds.data_vars)
print(ds.coords)

In [ ]:
for gname, grp in nc.groups.items():
    print("Group:", gname)
    print("  dimensions:", list(grp.dimensions.keys()))
    print("  variables:", list(grp.variables.keys()))

In [ ]:
nc = Dataset(f'{files[710]}', 'r')

In [ ]:
sss_var = nc.groups['Aquarius Data'].variables['SSS']

In [ ]:
sss_var.shape

In [ ]:
sss_ma = sss_var[:]
sss = sss_ma.filled(np.nan) 

In [ ]:
sss.shape

In [ ]:
np.count_nonzero(~np.isnan(sss))

In [ ]:
beam_cellon_var = nc.groups['Navigation'].variables['beam_clon']
beam_cellat_var = nc.groups['Navigation'].variables['beam_clat']

In [ ]:
beam_cellon_ma = beam_cellon_var[:]
beam_cellon = beam_cellon_ma.filled(np.nan) 

In [ ]:
beam_cellat_ma = beam_cellat_var[:]
beam_cellat = beam_cellat_ma.filled(np.nan) 

In [ ]:
beam_cellon.shape, beam_cellat.shape

In [ ]:
mask = ~np.isnan(sss)
idx = np.where(mask)[0]

In [ ]:
cellat = beam_cellat[idx]
cellon = beam_cellon[idx]
sss_ = sss[idx]

In [ ]:
cellat

In [ ]:
sss_

In [ ]:
fig = plt.figure(figsize=(10, 5))

m = Basemap(projection='cyl',
            llcrnrlat=-90, urcrnrlat=90,
            llcrnrlon=-180, urcrnrlon=180,
            resolution='c')

# Фон карты
m.drawcoastlines()
m.drawcountries()
m.drawmapboundary(fill_color='lightblue')
m.fillcontinents(color='cornsilk', lake_color='lightblue')

# Сетка
m.drawparallels(np.arange(-90, 91, 30), labels=[1,0,0,0], linewidth=0.2)
m.drawmeridians(np.arange(-180, 181, 60), labels=[0,0,0,1], linewidth=0.2)

# Переводим широту/долготу в координаты карты
x, y = m(cellon.ravel(), cellat.ravel())

# Наносим точки, цвет - по значениям
sc = m.scatter(x, y, c=sss_.ravel(), cmap='jet', s=50, edgecolor='k')

# Цветовая шкала
plt.colorbar(sc, orientation='vertical', label='Значение')

# plt.title()
plt.tight_layout()
plt.show()